# HOG result analysis

Comparison notebook for Lasserre level 1 vs. level 2 on HOG graph instances. It is based on the previous result-analysis notebook, but the plotting cells filter and label HOG graphs rather than line graphs.

In [2]:
from pathlib import Path
import json
import math
import pandas as pd


# -----------------------------
# Input / output paths
# -----------------------------
# Run this notebook either from the QAOA repository root or from the MasterThesis
# directory containing the QAOA repository.
project_root = Path.cwd()
if not (project_root / "Results").exists() and (project_root / "QAOA" / "Results").exists():
    project_root = project_root / "QAOA"

lasserre1_csv = project_root / "Results/logs/adam/qaoa_results_adam_20260617_172309_pid3959588.csv"
lasserre2_csv = project_root / "Results/logs/adam/qaoa_results_adam_20260617_173540_pid708337.csv"

output_csv = project_root / "aggregated_hog_lasserre1_vs_lasserre2.csv"


# -----------------------------
# Compact cell helpers
# -----------------------------
def make_json_safe(obj):
    """
    Recursively convert Python objects into JSON-safe objects.
    In particular, tuple dict keys like (0, 1) become strings like "0-1".
    """
    if isinstance(obj, dict):
        safe = {}
        for k, v in obj.items():
            if isinstance(k, tuple):
                key = "-".join(map(str, k))
            else:
                key = str(k)
            safe[key] = make_json_safe(v)
        return safe

    if isinstance(obj, tuple):
        return [make_json_safe(x) for x in obj]

    if isinstance(obj, list):
        return [make_json_safe(x) for x in obj]

    if isinstance(obj, (pd.Series, pd.Index)):
        return [make_json_safe(x) for x in obj.tolist()]

    if pd.isna(obj) if not isinstance(obj, (list, tuple, dict)) else False:
        return None

    return obj


def json_cell(obj) -> str:
    return json.dumps(make_json_safe(obj), sort_keys=False)


def stats_cell(stats: dict, digits: int | None = 10) -> str:
    payload = {
        "min": stats.get("min"),
        "mean": stats.get("mean"),
        "median": stats.get("median"),
        "max": stats.get("max"),
        "std": stats.get("std"),
        "count": stats.get("count", 0),
    }

    if digits is not None:
        for key in ("min", "mean", "median", "max", "std"):
            if payload[key] is not None and not pd.isna(payload[key]):
                payload[key] = round(float(payload[key]), digits)

    return json_cell(payload)


def ratio_stats_cell(metric_stats: dict, optimal_stats: dict, digits: int | None = 10) -> str:
    metric_raw = metric_stats.get("raw", [])
    optimal_raw = optimal_stats.get("raw", [])

    metric_clean = [float(x) for x in metric_raw if x is not None and not math.isnan(float(x))]
    optimal_clean = [float(x) for x in optimal_raw if x is not None and not math.isnan(float(x)) and float(x) != 0.0]

    ratios = []
    if metric_clean:
        # If there are more optimal values than metric values, use the first len(metric_clean)
        if len(optimal_clean) >= len(metric_clean):
            ratios = [m / optimal_clean[i] for i, m in enumerate(metric_clean)]
        # If the lengths differ and no direct matching is possible, fall back on mean ratios
        elif metric_stats.get("mean") is not None and optimal_stats.get("mean") not in (None, 0.0):
            ratios = [float(metric_stats["mean"]) / float(optimal_stats["mean"])]
    return stats_cell(_series_stats(ratios), digits=digits)


# -----------------------------
# Aggregate over intersecting sampled seeds/repeats
# -----------------------------
aggregated = aggregate_lasserre_qaoa_results(
    lasserre1_csv=lasserre1_csv,
    lasserre2_csv=lasserre2_csv,
    only_common=True,           # only benchmark settings that exist in both CSVs
    only_common_samples=True,   # only seeds/repeats that exist in both CSVs
    aggregation_mode="mean",
)


# -----------------------------
# One row per benchmark setting
# -----------------------------
rows = []

for benchmark_key, values in aggregated.items():
    graph_instance = values["graph_instance"]
    optimal_stats = values["optimal_value"]

    row = {
        # First column contains graph, settings, and actually used sample intersection.
        "graph_instance": json_cell(graph_instance),

        # Extra visible sample summary for convenience.
        "sample_summary": json_cell({
            "common_used": values["n_common_samples_used"],
            "lasserre1_used": values["n_rows_lasserre1"],
            "lasserre2_used": values["n_rows_lasserre2"],
            "lasserre1_total_available": values["n_rows_lasserre1_total_available"],
            "lasserre2_total_available": values["n_rows_lasserre2_total_available"],
        }),

        # Requested variables, each as a compact stats dict.
        "Optimal value from Vincenzo": stats_cell(values["optimal_value"]),

        # Step 1 energies (should be upper bounds)
        "Lasserre 1 Step 1 energy from relaxation (before GW rounding)": stats_cell(values["lasserre1_value_step1"]),
        "Lasserre 2 Step 1 energy from relaxation (before GW rounding)": stats_cell(values["lasserre2_value_step1"]),

        # Step 2 energies after GP rounding (for lasserre 1 this is also the final result)
        "Lasserre 1 Step 2 energy after GP rounding": stats_cell(
            values["lasserre1_value_after_gp_rounding_step2"]
        ),
        "Lasserre 2 Step 2 energy after GP rounding": stats_cell(
            values["lasserre2_value_after_gp_rounding_step2"]
        ),

        # Lasserre 2 final energy after GP + rotations
        "Lasserre 2 Final energy after GP + rotations": stats_cell(
            values["lasserre2_value_after_rotations_step10"]
        ),

        # QAOA results after using SDP vector as warm start
        "Lasserre 1 QAOA result after using SDP vector as warm start": stats_cell(values["lasserre1_qaoa_result"]),
        "Lasserre 2 QAOA result after using SDP vector as warm start": stats_cell(values["lasserre2_qaoa_result"]),

        # Optional approximation ratios, also compact dicts.
        "Lasserre 1 Step 1 SDP energy approx. ratio (before GW rounding)": ratio_stats_cell(
            values["lasserre1_value_step1"], optimal_stats
        ),
        "Lasserre 2 Step 1 SDP energy approx. ratio (before GW rounding)": ratio_stats_cell(
            values["lasserre2_value_step1"], optimal_stats
        ),
        "Lasserre 1 Step 2 GP GW-rounding energy approx. ratio": ratio_stats_cell(
            values["lasserre1_value_after_gp_rounding_step2"], optimal_stats
        ),
        "Lasserre 2 Step 2 GP GW-rounding energy approx. ratio": ratio_stats_cell(
            values["lasserre2_value_after_gp_rounding_step2"], optimal_stats
        ),
        "Lasserre 2 Final energy approx. ratio (after GP + rotations)": ratio_stats_cell(
            values["lasserre2_value_after_rotations_step10"], optimal_stats
        ),
        "Lasserre 1 QAOA approx. ratio": ratio_stats_cell(
            values["lasserre1_qaoa_result"], optimal_stats
        ),
        "Lasserre 2 QAOA approx. ratio": ratio_stats_cell(
            values["lasserre2_qaoa_result"], optimal_stats
        ),
    }

    rows.append(row)


comparison_df = pd.DataFrame(rows)

# Keep HOG graph rows only.
if not comparison_df.empty:
    comparison_df = comparison_df[
        comparison_df["graph_instance"].apply(lambda cell: json.loads(cell).get("graph_type") == "hog")
    ].copy()

# Stable ordering using values inside graph_instance.
def _sort_key(row):
    gi = json.loads(row["graph_instance"])
    return (
        gi.get("graph_type", ""),
        gi.get("hog_graph_index", -1),
        gi.get("#vertices", -1),
        gi.get("#edges", -1),
        gi.get("#qaoa_layers", -1),
    )


if not comparison_df.empty:
    comparison_df = (
        comparison_df
        .assign(_sort_key=comparison_df.apply(_sort_key, axis=1))
        .sort_values("_sort_key")
        .drop(columns="_sort_key")
        .reset_index(drop=True)
    )

comparison_df.to_csv(output_csv, index=False)

print(f"Wrote {len(comparison_df)} benchmark settings to {output_csv}")
comparison_df.head()

FileNotFoundError: [Errno 2] No such file or directory: '/Users/julian/University/MasterThesis/QAOA/Code/Results/logs/adam/qaoa_results_adam_20260617_172309_pid3959588.csv'

## HOG-specific comparison plots

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# -----------------------------
# HOG plotting helpers
# -----------------------------
def parse_json_cell(cell):
    if cell is None or pd.isna(cell):
        return {}
    if isinstance(cell, dict):
        return cell
    return json.loads(cell)


def extract_stat(cell, stat="mean"):
    payload = parse_json_cell(cell)
    value = payload.get(stat)
    if value is None:
        return np.nan
    try:
        return float(value)
    except Exception:
        return np.nan


def build_hog_plot_df(comparison_df: pd.DataFrame) -> pd.DataFrame:
    plot_df = comparison_df.copy()
    plot_df["_graph_info"] = plot_df["graph_instance"].apply(parse_json_cell)

    plot_df["graph_type"] = plot_df["_graph_info"].apply(lambda x: x.get("graph_type"))
    plot_df = plot_df[plot_df["graph_type"] == "hog"].copy()

    if plot_df.empty:
        raise ValueError("No HOG graph rows found in comparison_df.")

    plot_df["hog_graph_index"] = plot_df["_graph_info"].apply(lambda x: x.get("hog_graph_index"))
    plot_df["n_vertices"] = plot_df["_graph_info"].apply(lambda x: x.get("#vertices"))
    plot_df["n_edges"] = plot_df["_graph_info"].apply(lambda x: x.get("#edges"))
    plot_df["p"] = plot_df["_graph_info"].apply(lambda x: x.get("#qaoa_layers"))

    for col in ["hog_graph_index", "n_vertices", "n_edges", "p"]:
        plot_df[col] = pd.to_numeric(plot_df[col], errors="coerce")

    return plot_df


hog_plot_df = build_hog_plot_df(comparison_df)

print("HOG rows:", len(hog_plot_df))
print("p values:", sorted(hog_plot_df["p"].dropna().astype(int).unique().tolist()))
print("HOG graph index range:", int(hog_plot_df["hog_graph_index"].min()), "to", int(hog_plot_df["hog_graph_index"].max()))
print("vertex counts:", sorted(hog_plot_df["n_vertices"].dropna().astype(int).unique().tolist()))

hog_plot_df[["hog_graph_index", "n_vertices", "n_edges", "p", "sample_summary"]].head()

In [ ]:
# -----------------------------
# Coverage / sample summary by p and graph size
# -----------------------------
def sample_count(cell, key="common_used"):
    payload = parse_json_cell(cell)
    value = payload.get(key)
    try:
        return int(value)
    except Exception:
        return 0


coverage_df = hog_plot_df.copy()
coverage_df["common_samples_used"] = coverage_df["sample_summary"].apply(lambda c: sample_count(c, "common_used"))
coverage_df["lasserre1_used"] = coverage_df["sample_summary"].apply(lambda c: sample_count(c, "lasserre1_used"))
coverage_df["lasserre2_used"] = coverage_df["sample_summary"].apply(lambda c: sample_count(c, "lasserre2_used"))

coverage_summary = (
    coverage_df
    .groupby(["p", "n_vertices"], dropna=False)
    .agg(
        graph_rows=("hog_graph_index", "count"),
        min_common_samples=("common_samples_used", "min"),
        mean_common_samples=("common_samples_used", "mean"),
        max_common_samples=("common_samples_used", "max"),
        min_l1_samples=("lasserre1_used", "min"),
        min_l2_samples=("lasserre2_used", "min"),
    )
    .reset_index()
    .sort_values(["p", "n_vertices"])
)

coverage_summary

In [ ]:
# -----------------------------
# HOG approximation-ratio plots
# -----------------------------
save_hog_plots = False
hog_plot_output_dir = Path("hog_ratio_plots")
hog_plot_output_dir.mkdir(exist_ok=True)

ratio_types = [
    {
        "ratio_key": "step1",
        "title": "Step 1 SDP energy approx. ratio before GP/GW rounding",
        "columns": {
            "L1": "Lasserre 1 Step 1 SDP energy approx. ratio (before GW rounding)",
            "L2": "Lasserre 2 Step 1 SDP energy approx. ratio (before GW rounding)",
        },
    },
    {
        "ratio_key": "gp_rounding",
        "title": "Step 2 GP/GW rounding energy approx. ratio",
        "columns": {
            "L1": "Lasserre 1 Step 2 GP GW-rounding energy approx. ratio",
            "L2": "Lasserre 2 Step 2 GP GW-rounding energy approx. ratio",
        },
    },
    {
        "ratio_key": "rotations",
        "title": "Final energy approx. ratio after GP + rotations",
        "columns": {
            "L2": "Lasserre 2 Final energy approx. ratio (after GP + rotations)",
        },
    },
    {
        "ratio_key": "qaoa",
        "title": "QAOA result approx. ratio",
        "columns": {
            "L1": "Lasserre 1 QAOA approx. ratio",
            "L2": "Lasserre 2 QAOA approx. ratio",
        },
    },
]

# Extract JSON stats into numeric columns.
stats = ["min", "mean", "median", "max", "std"]
for ratio in ratio_types:
    for level, col_name in ratio["columns"].items():
        for stat in stats:
            hog_plot_df[f"{col_name}_{stat}"] = hog_plot_df[col_name].apply(lambda cell, s=stat: extract_stat(cell, s))


def aggregate_by_vertices(df: pd.DataFrame, col_name: str) -> pd.DataFrame:
    """
    Aggregate HOG instances by number of vertices for one ratio column.
    The mean/median are averaged across HOG instances. The shaded ranges use:
    - min/max: min/max across the per-row min/max stats;
    - std: average per-row std around the averaged mean.
    """
    rows = []
    for n, group in df.groupby("n_vertices"):
        rows.append({
            "n_vertices": int(n),
            "min": group[f"{col_name}_min"].min(),
            "mean": group[f"{col_name}_mean"].mean(),
            "median": group[f"{col_name}_median"].mean(),
            "max": group[f"{col_name}_max"].max(),
            "std": group[f"{col_name}_std"].mean(),
            "count": len(group),
        })
    return pd.DataFrame(rows).sort_values("n_vertices")


for ratio in ratio_types:
    for p_value in sorted(hog_plot_df["p"].dropna().astype(int).unique()):
        fig, ax = plt.subplots(figsize=(12, 7))
        has_data = False

        p_df = hog_plot_df[hog_plot_df["p"] == p_value].copy()

        for level, col_name in ratio["columns"].items():
            available = p_df[p_df[col_name].notna()].copy()
            if available.empty:
                continue

            agg = aggregate_by_vertices(available, col_name)
            if agg.empty:
                continue

            x = agg["n_vertices"].to_numpy(dtype=int)
            y_min = agg["min"].to_numpy(dtype=float) * 100.0
            y_mean = agg["mean"].to_numpy(dtype=float) * 100.0
            y_median = agg["median"].to_numpy(dtype=float) * 100.0
            y_max = agg["max"].to_numpy(dtype=float) * 100.0
            y_std = agg["std"].fillna(0.0).to_numpy(dtype=float) * 100.0

            linestyle = (0, (6, 3)) if level == "L1" else "-"
            marker = "o" if level == "L1" else "s"

            ax.fill_between(x, y_min, y_max, alpha=0.12, label=f"{level} min-max range")
            ax.fill_between(x, y_mean - y_std, y_mean + y_std, alpha=0.18, label=f"{level} mean ± std")
            ax.plot(x, y_mean, linestyle=linestyle, marker=marker, linewidth=2.3, label=f"{level} mean")
            ax.plot(x, y_median, linestyle=linestyle, marker="^", linewidth=1.8, label=f"{level} median")

            has_data = True

        if not has_data:
            plt.close(fig)
            continue

        ax.axhline(100.0, linestyle=":", linewidth=1.5, color="black", label="Optimal = 100%")
        ax.set_xlabel("Number of vertices")
        ax.set_ylabel("Approximation ratio (%)")
        ax.set_title(f"HOG graphs, p={p_value}: {ratio['title']}")
        ax.set_ylim(bottom=0)
        ax.set_xticks(sorted(p_df["n_vertices"].dropna().astype(int).unique()))
        ax.grid(True, alpha=0.3)

        handles, labels = ax.get_legend_handles_labels()
        unique = {}
        for h, label in zip(handles, labels):
            if label not in unique:
                unique[label] = h
        ax.legend(unique.values(), unique.keys(), loc="best", handlelength=3.0)

        plt.tight_layout()

        if save_hog_plots:
            fig.savefig(hog_plot_output_dir / f"hog_{ratio['ratio_key']}_p{p_value}.png", dpi=200)

        plt.show()

In [ ]:
# -----------------------------
# Optional: plot QAOA ratio by exact HOG graph index
# -----------------------------
# This is noisier than the vertex-count aggregation above, but useful for spotting individual hard graphs.

save_hog_index_plots = False

qaoa_columns = {
    "L1": "Lasserre 1 QAOA approx. ratio",
    "L2": "Lasserre 2 QAOA approx. ratio",
}

for p_value in sorted(hog_plot_df["p"].dropna().astype(int).unique()):
    p_df = hog_plot_df[hog_plot_df["p"] == p_value].sort_values("hog_graph_index").copy()

    fig, ax = plt.subplots(figsize=(14, 6))
    has_data = False

    for level, col_name in qaoa_columns.items():
        if col_name not in p_df.columns:
            continue

        df_level = p_df[p_df[col_name].notna()].copy()
        if df_level.empty:
            continue

        x = df_level["hog_graph_index"].to_numpy(dtype=int)
        y = df_level[col_name].apply(lambda c: extract_stat(c, "mean")).to_numpy(dtype=float) * 100.0
        y_std = df_level[col_name].apply(lambda c: extract_stat(c, "std")).fillna(0.0).to_numpy(dtype=float) * 100.0

        linestyle = (0, (6, 3)) if level == "L1" else "-"
        ax.plot(x, y, linestyle=linestyle, linewidth=1.8, label=f"{level} mean")
        ax.fill_between(x, y - y_std, y + y_std, alpha=0.12, label=f"{level} mean ± std")

        has_data = True

    if not has_data:
        plt.close(fig)
        continue

    ax.axhline(100.0, linestyle=":", linewidth=1.5, color="black", label="Optimal = 100%")
    ax.set_xlabel("HOG graph index")
    ax.set_ylabel("QAOA approximation ratio (%)")
    ax.set_title(f"HOG graphs, p={p_value}: QAOA ratio by graph index")
    ax.set_ylim(bottom=0)
    ax.grid(True, alpha=0.25)

    handles, labels = ax.get_legend_handles_labels()
    unique = {}
    for h, label in zip(handles, labels):
        if label not in unique:
            unique[label] = h
    ax.legend(unique.values(), unique.keys(), loc="best", handlelength=3.0)

    plt.tight_layout()

    if save_hog_index_plots:
        fig.savefig(hog_plot_output_dir / f"hog_qaoa_by_index_p{p_value}.png", dpi=200)

    plt.show()